# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [18]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
    'table': ["Customers", "Orders"],
    'definition': ["""
        CREATE TABLE Customers (
            customer_id INT PRIMARY KEY,
            first_name VARCHAR(50),
            last_name VARCHAR(50),
            email VARCHAR(100),
            phone VARCHAR(20),
            address VARCHAR(255),
            city VARCHAR(50),
            state VARCHAR(50),
            zip_code VARCHAR(10),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
    """,
    """
        CREATE TABLE Orders (
            order_id INT PRIMARY KEY,
            customer_id INT,
            order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            total_amount DECIMAL(10, 2),
            status VARCHAR(20),
            FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
        );
        """]
}
 # ENTER A TABLE DEFINITATIONS HERE
df = pd.DataFrame(data)
print(df)
print(df.loc[1, 'definition'])

       table                                         definition
0  Customers  \n        CREATE TABLE Customers (\n          ...
1     Orders  \n        CREATE TABLE Orders (\n            o...

        CREATE TABLE Orders (
            order_id INT PRIMARY KEY,
            customer_id INT,
            order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            total_amount DECIMAL(10, 2),
            status VARCHAR(20),
            FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
        );
        


In [19]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [20]:
print(text_tables)

Customers: 
        CREATE TABLE Customers (
            customer_id INT PRIMARY KEY,
            first_name VARCHAR(50),
            last_name VARCHAR(50),
            email VARCHAR(100),
            phone VARCHAR(20),
            address VARCHAR(255),
            city VARCHAR(50),
            state VARCHAR(50),
            zip_code VARCHAR(10),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
    
Orders: 
        CREATE TABLE Orders (
            order_id INT PRIMARY KEY,
            customer_id INT,
            order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            total_amount DECIMAL(10, 2),
            status VARCHAR(20),
            FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
        );
        


In [9]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [16]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="How many customers are there in the database?")
print(pqt1)


Given the following tables and their content definitions,
###Tables
Customers: 
        CREATE TABLE Customers (
            customer_id INT PRIMARY KEY,
            first_name VARCHAR(50),
            last_name VARCHAR(50),
            email VARCHAR(100),
            phone VARCHAR(20),
            address VARCHAR(255),
            city VARCHAR(50),
            state VARCHAR(50),
            zip_code VARCHAR(10),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
    

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
How many customers are there in the database?



In [21]:
print(return_OAI(pqt1))

```json
{
    "tables": ["Customers"]
}
```


In [22]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="what is the email address of the customer with the first name 'John' amd last name ' Doe'?")

In [23]:
print(return_OAI(pqt3))

{
    "tables": ["Customers"]
}


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Version 1

In [26]:
test_questions = [
    "What is the total number of orders placed in 2025?",
    "List the names of customers who placed orders with a total amount greater than $100.",
    "What is the average order amount for each customer in California?"
]
for question in test_questions:
    print(f"\nTesting question: {question}")
    pqt = prompt_question_tables.format(tables=text_tables, question=question)
    response = return_OAI(pqt)
    print(f"Response:\n{response}\n")


Testing question: What is the total number of orders placed in 2025?
Response:
{
    "tables": ["Orders"]
}


Testing question: List the names of customers who placed orders with a total amount greater than $100.
Response:
```json
{
    "tables": ["Customers", "Orders"]
}
```


Testing question: What is the average order amount for each customer in California?
Response:
{
    "tables": ["Customers", "Orders"]
}



## Observations
* For "what is the total number of orders placed in 2025?" 
    * question, The expected response is correct, as only the Orders table is needed.
    * If the model includes Customers or other nonexistent tables, it would indicate a hallucination or misunderstanding of the schema
    * potential Issue might be that GPT-3.5 might misinterpret “2025” if not explicitly trained to handle date filters, but the table selection should still be correct
* For "ist the names of customers who placed orders with a total amount greater than $100?"
    * This question requires customer names (first_name, last_name from Customers) and order details (total_amount from Orders), necessitating a JOIN on customer_id.
    * The expected response is correct, as both tables are required
    * Potential issue assumption is the model might omit one table or include irrelevant ones if it misinterprets the question’s intent.
* For "What is the average order amount for each customer in California?"
    * This question requires state from Customers (to filter for California) and total_amount from Orders (to compute the average), linked via customer_id.
    * The expected response is correct, as both tables are needed.
    * Potential Issue assumption is GPT-3.5 might struggle with aggregation logic if asked for the SQL query, but table selection should be straightforward.

## Version 2

In [28]:
test_questions = [
    "What is the weather forecast for tomorrow?",
    "Which customers have not placed any orders?"
]
for question in test_questions:
    print(f"\nTesting question: {question}")
    pqt = prompt_question_tables.format(tables=text_tables, question=question)
    response = return_OAI(pqt)
    print(f"Response:\n{response}\n")


Testing question: What is the weather forecast for tomorrow?
Response:
```json
{
    "tables": []
}
```


Testing question: Which customers have not placed any orders?
Response:
```json
{
    "tables": ["Customers", "Orders"]
}
```



## Observations
* For "What is the weather forecast for tomorrow?"
    * This question is irrelevant to the provided tables, as neither Customers nor Orders contains weather-related data.
    * The expected response is an empty table list, indicating no relevant tables.
* For "Which customers have not placed any orders?"
    * This question requires a LEFT JOIN to identify customers (Customers table) who have no matching orders (Orders table), e.g., SELECT c.first_name, c.last_name FROM Customers c LEFT JOIN Orders o ON c.customer_id = o.customer_id WHERE o.order_id IS NULL
    * The expected response is correct, as both tables are needed to check for non-matching records.
    * The model might misinterpret the question and suggest only Customers or include irrelevant tables.

In [27]:
prompt_sql_query = """
Given the following tables and their content definitions,
###Tables
{tables}
Write a SQL query to address the user's question below.
###User Question:
{question}
"""
question = "List the names of customers who placed orders with a total amount greater than $100."
pqt_sql = prompt_sql_query.format(tables=text_tables, question=question)
response = return_OAI(pqt_sql)
print(f"SQL Query for '{question}':\n{response}\n")

SQL Query for 'List the names of customers who placed orders with a total amount greater than $100.':
```sql
SELECT c.first_name, c.last_name
FROM Customers c
JOIN Orders o ON c.customer_id = o.customer_id
WHERE o.total_amount > 100;
```



## Lessons Learnt
* Table Selection Accuracy:
    * GPT-3.5-Turbo excels at selecting relevant tables for straightforward queries when provided with clear schema definitions (e.g., pqt1 and pqt3 correctly identified Customers).
    * For join-based queries (e.g., Questions 4, 5, 7), the model typically recognizes the need for multiple tables when the question explicitly references columns from both, but you should verify if it consistently includes both Customers and Orders.
* Handling Irrelevant Queries:
    * The model’s ability to handle irrelevant questions (e.g., Question 6) is critical. If it returns an empty list ([]), it demonstrates robustness. 
    * If it hallucinates nonexistent tables, it highlights a limitation in context understanding, a common issue with language models.
* Prompt Design Importance:
    * The structured prompt (with clear sections for tables and questions) improved the model’s ability to provide focused, JSON-formatted responses. Poorly designed prompts could lead to less reliable outputs.
* Hallucination Risks:
    * The model may hallucinate table names or columns for out-of-scope questions (e.g., Question 6) or misinterpret complex queries. This underscores the need for validation when using AI for database tasks.
* Schema Dependency:
    * The model’s performance heavily relies on clear, detailed schema definitions. The provided CREATE TABLE statements helped it map questions to tables accurately, but ambiguous or incomplete schemas could lead to errors.
* SQL Query Generation Challenges:
    * Generating full SQL queries (as tested optionally) is more error-prone than table selection. GPT-3.5 may produce syntactically correct queries but could fail on complex logic (e.g., aggregations, LEFT JOIN for Question 7) or use incorrect column names if the schema isn’t fully understood.